# Running the whole network

In [ ]:
import time

import pandas as pd

from recon import check, db, jobs, queue
from recon.config import settings
from recon.execution import SepexClient

runner = SepexClient(base_url=settings.sepex_url)


def tally():
    return db.one("""
        SELECT count(*) FILTER (WHERE state = 'finished')           AS finished,
               count(*) FILTER (WHERE state = 'awaiting_downstream') AS waiting,
               count(*) FILTER (WHERE state = 'awaiting_inputs')    AS awaiting,
               count(*) FILTER (WHERE state = 'in_flight')          AS in_flight,
               count(*) FILTER (WHERE state = 'resting')            AS resting,
               count(*) FILTER (WHERE state = 'halted')             AS halted,
               count(*)                                             AS total
        FROM reach_status""")


print(f"sepex    {settings.sepex_url}")
print(f"run job  {check.RUN_ND_PROCESSES[('lisflood', check.gpu_available())]}")
print(tally())

## Settings

In [ ]:
RUN_FOREVER = False
INTERVAL = 8

## The loop

In [ ]:
started = time.time()
passes = 0
try:
    while True:
        passes += 1
        for outcome in jobs.poll_in_flight(runner):
            if outcome["status"] in ("succeeded", "failed"):
                print(f"    reach {outcome['reach_id']} {outcome['step']}: "
                      f"{outcome['status']} - {outcome['action']}")

        submitted = 0
        for row in queue.due_reaches():
            if check.run_check(row["reach_id"], runner).submitted_ref:
                submitted += 1

        now = tally()
        print(f"[{time.time() - started:5.0f}s] pass {passes:3d}  "
              f"finished {now['finished']:>3}/{now['total']}  waiting {now['waiting']:>3}  "
              f"awaiting {now['awaiting']:>2}  in flight {now['in_flight']:>2}  "
              f"resting {now['resting']:>2}  halted {now['halted']:>2}  submitted {submitted}")

        steady = now["in_flight"] == 0 and submitted == 0 and passes > 1
        if steady and not RUN_FOREVER:
            print(f"\nsteady after {passes} passes in {time.time() - started:.0f}s")
            break
        time.sleep(INTERVAL)
except KeyboardInterrupt:
    print("\nstopped by hand; nothing lost — in-flight jobs are recorded in the database")

## Where the network stands

In [ ]:
display(pd.DataFrame(db.query(
    "SELECT state, count(*) AS reaches FROM reach_status GROUP BY state ORDER BY reaches DESC")))

display(pd.DataFrame(db.query("""
    SELECT rn.is_terminal,
           count(*)                                              AS reaches,
           count(*) FILTER (WHERE rs.model_id IS NOT NULL)        AS with_model,
           count(*) FILTER (WHERE rs.nd_materialized)             AS with_nd_library,
           sum(rs.nd_discharges)                                  AS scenarios
    FROM reach_status rs JOIN reach_network rn USING (reach_id)
    GROUP BY 1 ORDER BY 1 DESC""")))

## The wait graph

In [ ]:
pd.DataFrame(db.query("""
    SELECT p.blocked_on_reach_id                    AS waits_on,
           ds.state                                 AS its_state,
           count(*)                                 AS reaches_waiting
    FROM reach_status rs
    JOIN reach_processing p USING (reach_id)
    JOIN reach_status ds ON ds.reach_id = p.blocked_on_reach_id
    WHERE rs.state = 'awaiting_downstream'
    GROUP BY 1, 2 ORDER BY reaches_waiting DESC"""))

## Anything parked

In [ ]:
halted = db.query("SELECT reach_id, consecutive_failures, halted_at, last_error "
                  "FROM reach_processing WHERE halted")
if halted:
    display(pd.DataFrame(halted))
    print("\nafter fixing the cause:  processing.clear_halt(reach_id)")
else:
    print("nothing halted")